# Computational Theory Assessment
# Student ID-G00413222
# Student Name-Fawas Ajani

## Introduction

This notebook contains my work for the Computational Theory assessment. The main focus of the assessment is the SHA-256 hashing algorithm, following the NIST Secure Hash Standard (FIPS 180-4).

The work is divided into several problems. I will first look at how SHA-256 data can be represented in Python before implementing the required bitwise operations, constants, message padding, compression function, and finally the complete SHA-256 algorithm.

Each section includes explanations and Python code, along with tests where appropriate.

## Project Structure

- **Problem 1**: Representing SHA-256 data, including 32-bit words, input messages, 512-bit blocks, the 256-bit hash, and big-endian integers.
- **Problem 2**: Implementing the SHA-256 bitwise functions using NumPy.
- **Problem 3**: Generating the SHA-256 constants from prime numbers and their cube roots.
- **Problem 4**: Padding and parsing input messages into 512-bit blocks.
- **Problem 5**: Implementing the SHA-256 compression function.
- **Problem 6**: Combining the previous components to implement and test the complete SHA-256 algorithm.

## Libraries

This assessment uses NumPy for working with 32-bit unsigned integers and arrays. These are useful for representing the words and values used by SHA-256.

In [5]:
# Import NumPy
import numpy as np

## Problem 1: Representing SHA-256 Data

SHA-256 works with fixed-size pieces of data, so it is important to represent these values correctly before implementing the algorithm. This problem looks at 32-bit words, sequences of 32-bit words, input messages, 512-bit blocks, the 256-bit hash value, and big-endian integers.


### 32-bit Words

SHA-256 processes data using 32-bit words. A 32-bit unsigned integer can represent values from \(0\) to \(2^{32}-1\). NumPy's `uint32` type can be used to represent these values explicitly.

Using a fixed-width 32-bit type is important because the SHA-256 algorithm defines its operations using 32-bit words.


In [6]:
# Create a 32-bit unsigned integer
word = np.uint32(42)

print("Value:", word)
print("Type:", type(word))
print("Maximum value:", np.iinfo(np.uint32).max)

Value: 42
Type: <class 'numpy.uint32'>
Maximum value: 4294967295


In [7]:
# Test the maximum value of a 32-bit unsigned integer
maximum_word = np.uint32(2**32 - 1)

print("Maximum 32-bit value:", maximum_word)
print("Hexadecimal:", hex(int(maximum_word)))

Maximum 32-bit value: 4294967295
Hexadecimal: 0xffffffff


The test shows that a 32-bit unsigned integer can store values up to \(2^{32}-1\), which is `4294967295`. The hexadecimal representation `0xffffffff` shows all 32 bits set to 1. This confirms the range of the `uint32` representation being used for SHA-256 words.


### Sequences of 32-bit Words

SHA-256 also works with sequences of 32-bit words. A sequence can be represented using a NumPy array with the `uint32` data type. A 512-bit SHA-256 message block contains 16 words, with each word containing 32 bits.


In [8]:
# Create a sequence of 32-bit words
words = np.array([0, 1, 2, 3], dtype=np.uint32)

print("Words:", words)
print("Number of words:", len(words))
print("Data type:", words.dtype)

Words: [0 1 2 3]
Number of words: 4
Data type: uint32


A SHA-256 message block is 512 bits in size. Since each word contains 32 bits, one block contains 16 words because \(512 \div 32 = 16\). Therefore, a complete block can be represented as a NumPy array containing 16 `uint32` values.


In [9]:
# Create a 512-bit block as 16 32-bit words
block = np.zeros(16, dtype=np.uint32)

print("Block:", block)
print("Number of words:", len(block))
print("Data type:", block.dtype)

Block: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Number of words: 16
Data type: uint32


The result contains 16 words, and each word is a 32-bit unsigned integer. Therefore, the total size of the block is \(16 \times 32 = 512\) bits. This matches the 512-bit message block size used by SHA-256.


### Input Messages

The input to SHA-256 is a message made up of bytes. In Python, a message can be represented using the `bytes` data type. This allows text or other data to be stored as a sequence of values between 0 and 255.

For example, the text `"Hello"` can be encoded as bytes using UTF-8.


In [10]:

# Represent a message as bytes
message = b"Hello"

print("Message:", message)
print("Length in bytes:", len(message))
print("Byte values:", list(message))

Message: b'Hello'
Length in bytes: 5
Byte values: [72, 101, 108, 108, 111]


The message is stored as 5 bytes because `"Hello"` contains 5 characters in this example. Each byte has a value between 0 and 255. Representing the input as bytes is useful because SHA-256 processes the message as a sequence of bits and bytes before dividing it into 512-bit blocks.


### 256-bit Hash
SHA-256 produces a hash value that is 256 bits long. In Python, the hash can be represented as a sequence of bytes or as a hexadecimal string. A 256-bit hash contains 32 bytes, and each byte is represented by two hexadecimal characters, giving a total of 64 hexadecimal characters.

In [11]:
# Represent a 256-bit hash as 32 bytes
hash_bytes = np.zeros(32, dtype=np.uint8)

print("Hash length in bytes:", len(hash_bytes))
print("Hash length in bits:", len(hash_bytes) * 8)
print("Data type:", hash_bytes.dtype)

Hash length in bytes: 32
Hash length in bits: 256
Data type: uint8


The example contains 32 bytes, and each byte contains 8 bits. Therefore, \(32 \times 8 = 256\) bits. This shows how the fixed 256-bit size of a SHA-256 hash can be represented using a NumPy byte array.


In [12]:
# Represent a SHA-256 hash as a hexadecimal string
hash_hex = "2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824"

print("Hash:", hash_hex)
print("Number of hexadecimal characters:", len(hash_hex))

Hash: 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
Number of hexadecimal characters: 64


The hexadecimal string contains 64 characters. Each hexadecimal character represents 4 bits, so the total is \(64 \times 4 = 256\) bits. This demonstrates how a SHA-256 hash can be displayed as a hexadecimal string.


### Big-Endian Integers

Big-endian byte order means that the most significant byte is stored first. SHA-256 uses big-endian byte order when converting message bytes into 32-bit words.This means that the order of the bytes affects the integer value that is produced. Python's `int.from_bytes()` function can demonstrate this by interpreting a sequence of bytes as an integer.





In [13]:
# Convert four bytes into a 32-bit integer
byte_values = bytes([0x12, 0x34, 0x56, 0x78])

big_endian_value = int.from_bytes(byte_values, byteorder="big")

print("Bytes:", byte_values.hex())
print("Integer:", big_endian_value)
print("Hexadecimal:", hex(big_endian_value))

Bytes: 12345678
Integer: 305419896
Hexadecimal: 0x12345678


The four bytes are interpreted in the order `12 34 56 78`, with `12` as the most significant byte. The resulting integer is `0x12345678`. This demonstrates big-endian byte order, which is used when interpreting the bytes of a SHA-256 message block as 32-bit words.
